# Reproducing *Short Interest Trend Prediction with Large Language Models* (Xiao et al., ICIAI 2024)

**Stage 1 reproduction notebook — LLM Applications in Finance**

What this notebook does, end to end:
1. Downloads the public raw data (Benzinga headlines from Kaggle + FINRA files) — or loads a prebuilt labeled file.
2. Builds the labeled dataset exactly as described in Section 3 of the paper.
3. Creates the stratified 80/20 split (Section 4) and reports the same dataset statistics as the paper.
4. Runs the majority-class baseline (plus a random baseline the paper does not report).
5. Prompts an instruction-tuned LLM with the paper's Figure 4 prompt in 0/1/5/10-shot settings.
6. Scores predictions with per-class F1 and support-weighted F1 (Table 1 format) and saves everything.

**Two label sets** (pick with `LABEL_SET`):
* `"regsho_volume"` — **paper-faithful.** The paper's footnote 2 links to FINRA's *Daily Short Sale Volume* files, so its
  label is really the sign of the change in daily **short-sale volume** between trading days t−1 and t+1.
  This is *not* short interest (open short positions); we keep the paper's definition only so the numbers are comparable.
* `"short_interest"` — **true short interest.** FINRA consolidated short interest (open short positions, published twice a
  month). Label = sign of the change between the two reports that bracket the settlement of the news day's trades.
  Public FINRA history starts 2017-12-29, so this set covers 2018-01 → 2020-06 only.

The original authors did not release code or a processed dataset (none found as of 2026-10-14); everything below is rebuilt
from the public sources the paper names.

## 0. Configuration
Edit this cell only. Defaults are sized for a **free Colab T4 GPU** (Runtime → Change runtime type → T4 GPU).
With the defaults (FLAN-T5-XL, 500 test titles, 0/1/5/10-shot) the run is expected to take well under an hour on a T4;
that is an estimate, not a measurement. N_TEST=500 with SEED=42 selects exactly the same 500 test titles as the
FLAN-T5-large results reported in the Stage 1 report, so the numbers are directly comparable.

In [ ]:
import os

# ---- data ----
DATA_MODE = os.environ.get("DATA_MODE", "prebuilt")     # "prebuilt" = load sitp_dataset.parquet (fast); "rebuild" = download raw data & rebuild (~10-15 min)
LABEL_SET = os.environ.get("LABEL_SET", "regsho_volume")  # "regsho_volume" (paper-faithful) or "short_interest" (true SI)
WORK_DIR = os.environ.get("WORK_DIR", "/content/sitp")
# To survive Colab disconnects, keep everything on Google Drive instead (predictions resume from the last saved batch):
# from google.colab import drive; drive.mount("/content/drive"); WORK_DIR = "/content/drive/MyDrive/sitp"
PREBUILT_PATH = os.environ.get("PREBUILT_PATH", f"{WORK_DIR}/sitp_dataset.parquet")

# ---- model ----
# Paper models: "google/flan-t5-xxl" (11B), "google/flan-ul2" (20B), "meta-llama/Llama-2-7b-chat-hf" (7B, gated).
#   Free T4 (15 GB):   google/flan-t5-xl (3B) in fp16                     <- default
#                      meta-llama/Llama-2-7b-chat-hf with LOAD_8BIT=1 (needs Meta approval + HF_TOKEN secret)
#   A100 (Colab Pro):  google/flan-t5-xxl (bf16) or google/flan-ul2 with LOAD_8BIT=1
MODEL_NAME = os.environ.get("MODEL_NAME", "google/flan-t5-xl")
LOAD_8BIT = os.environ.get("LOAD_8BIT", "0") == "1"      # paper loads models in 8-bit (bitsandbytes)
MAX_INPUT_LEN = 2048                                     # paper: truncation=True, max input length 2048
SCORING = os.environ.get("SCORING", "generate")          # "generate" (paper: complete the prompt) or "rank" (compare option log-likelihoods)

# ---- experiment ----
SHOTS = [int(s) for s in os.environ.get("SHOTS", "0,1,5,10").split(",")]
N_TEST = int(os.environ.get("N_TEST", "500"))    # stratified sample of the 20% test split (full test set is ~263k titles)
BATCH_SIZE = int(os.environ.get("BATCH_SIZE", "8"))
SEED = 42
HF_TOKEN = os.environ.get("HF_TOKEN")            # only needed for gated models (Llama-2)
if HF_TOKEN is None:
    try:                                         # Colab: add HF_TOKEN under the key icon ("Secrets") in the left sidebar
        from google.colab import userdata  # type: ignore
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
os.makedirs(WORK_DIR, exist_ok=True)
RUN_TAG = f"{LABEL_SET}__{MODEL_NAME.split('/')[-1]}__{SCORING}__n{N_TEST}"
OUT_DIR = f"{WORK_DIR}/results/{RUN_TAG}"
os.makedirs(OUT_DIR, exist_ok=True)
print("Run:", RUN_TAG)

## 1. Install dependencies
(In Colab, torch is preinstalled. `bitsandbytes` is only needed when `LOAD_8BIT=True`.)

In [ ]:
import subprocess, sys
def pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)
try:
    import transformers, sentencepiece, accelerate, pyarrow  # noqa: F401
except ImportError:
    pip("transformers", "sentencepiece", "accelerate", "pyarrow")
if LOAD_8BIT:
    pip("bitsandbytes")

import glob, json, time, zipfile, datetime as dt, concurrent.futures as cf
import numpy as np
import pandas as pd
import requests
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix
print("torch", torch.__version__, "| transformers", transformers.__version__, "| CUDA:", torch.cuda.is_available())

## 2. Download raw data (skipped when `DATA_MODE="prebuilt"`)
* **News titles** — Kaggle *Massive Stock News Analysis DB for NLP/Backtests* (Benzinga headlines 2009–2020), published by the
  author of the Benzinga scraping tool the paper cites in footnote 4. All four Figure 1 headlines appear verbatim in its
  `analyst_ratings_processed.csv`. Public datasets download without a Kaggle key.
* **FINRA Reg SHO daily short-sale volume** — `cdn.finra.org/equity/regsho/daily/` (the files in the paper's footnote 2).
* **FINRA consolidated short interest** — `cdn.finra.org/equity/otcmarket/biweekly/` (true short interest, 2017-12 onward).

In [ ]:
RAW = f"{WORK_DIR}/raw"
KAGGLE_URL = "https://www.kaggle.com/api/v1/datasets/download/miguelaenlle/massive-stock-news-analysis-db-for-nlpbacktests"
REGSHO_URL = "https://cdn.finra.org/equity/regsho/daily/{fac}shvol{d}.txt"
SI_URL = "https://cdn.finra.org/equity/otcmarket/biweekly/shrt{d}.csv"
FACILITIES = ["FNSQ", "FNYX", "FNQC", "FNRA"]   # Nasdaq TRF, NYSE TRF, Chicago TRF, ADF (NMS stocks; ORF is OTC-only)
_session = requests.Session()

def fetch(url, path, retries=4):
    if os.path.exists(path):
        return 200
    for a in range(retries):
        try:
            r = _session.get(url, timeout=60)
            if r.status_code == 200:
                with open(path, "wb") as f:
                    f.write(r.content)
            return r.status_code
        except requests.RequestException:
            time.sleep(2 * (a + 1))
    return -1

def download_raw():
    os.makedirs(f"{RAW}/kaggle", exist_ok=True); os.makedirs(f"{RAW}/regsho", exist_ok=True); os.makedirs(f"{RAW}/si", exist_ok=True)
    news_csv = f"{RAW}/kaggle/analyst_ratings_processed.csv"
    if not os.path.exists(news_csv):
        print("Downloading Kaggle headlines (~220 MB)...")
        with _session.get(KAGGLE_URL, stream=True, timeout=600) as r:
            r.raise_for_status()
            with open(f"{RAW}/kaggle/archive.zip", "wb") as f:
                for chunk in r.iter_content(1 << 20):
                    f.write(chunk)
        with zipfile.ZipFile(f"{RAW}/kaggle/archive.zip") as z:
            z.extract("analyst_ratings_processed.csv", f"{RAW}/kaggle")
    days, d = [], dt.date(2009, 7, 27)
    while d <= dt.date(2020, 7, 15):
        if d.weekday() < 5:
            days.append(d.strftime("%Y%m%d"))
        d += dt.timedelta(days=1)
    jobs = [(REGSHO_URL.format(fac=f, d=x), f"{RAW}/regsho/{f}{x}.txt") for x in days for f in FACILITIES]
    print(f"Downloading up to {len(jobs)} FINRA daily short-sale volume files (non-trading days return 403)...")
    with cf.ThreadPoolExecutor(32) as ex:
        codes = list(ex.map(lambda j: fetch(*j), jobs))
    print("  files present:", sum(c == 200 for c in codes))
    r = _session.post("https://api.finra.org/data/group/otcMarket/name/consolidatedShortInterest",
                      json={"limit": 1000, "fields": ["settlementDate"],
                            "domainFilters": [{"fieldName": "symbolCode", "values": ["AAPL"]}]},
                      headers={"Accept": "application/json"}, timeout=60)
    sdates = sorted({x["settlementDate"].replace("-", "") for x in r.json()})
    sdates = [x for x in sdates if "20171201" <= x <= "20200831"]
    for x in sdates:
        fetch(SI_URL.format(d=x), f"{RAW}/si/shrt{x}.csv")
    print("  short-interest settlement files:", len(sdates))

if DATA_MODE == "rebuild" and not os.path.exists(PREBUILT_PATH):
    download_raw()

## 3. Build the labeled dataset (paper Section 3)
* Keep titles that are pure ASCII and ≤ 50 tokens (paper filters). Tokens are counted with the FLAN-T5 tokenizer, which
  reproduces the paper's average title length far better than whitespace splitting (see Section 4 check below).
* Match title ↔ ticker ↔ news date (US/Eastern calendar date of the timestamp).
* **regsho_volume:** S_t = daily short-sale volume summed over FINRA facilities; label = increase if S_{t+1} − S_{t−1} > 0,
  decrease if < 0, dropped if = 0 or missing. t−1 / t+1 are the trading days before / after the news date.
  (The paper defines short interest as shares shorted / shares outstanding; shares outstanding is constant over two days,
  so the sign of the change equals the sign of the change in shares shorted.)
* **short_interest:** trades on news day t settle at T+2. Label = sign(SI[s_after] − SI[s_before]) where s_after is the first
  FINRA settlement date ≥ settle(t) and s_before the report before it.

In [ ]:
from transformers import AutoTokenizer

def build_dataset():
    news = pd.read_csv(f"{RAW}/kaggle/analyst_ratings_processed.csv", usecols=["title", "date", "stock"], dtype=str).dropna()
    news["title"] = news["title"].str.strip()
    ts = pd.to_datetime(news["date"], utc=True, errors="coerce", format="ISO8601")
    news = news[ts.notna()].copy()
    news["news_date"] = ts[ts.notna()].dt.tz_convert("America/New_York").dt.strftime("%Y%m%d").astype(int)
    news = news[news["title"].map(str.isascii)]
    tok = AutoTokenizer.from_pretrained("google/flan-t5-base")
    titles = news["title"].tolist()
    news["n_tokens"] = [len(x) for i in range(0, len(titles), 100_000)
                        for x in tok(titles[i:i + 100_000], add_special_tokens=False)["input_ids"]]
    news = news[(news["n_tokens"] > 0) & (news["n_tokens"] <= 50)]
    news = news.drop_duplicates(["title", "stock", "news_date"]).rename(columns={"stock": "ticker"})
    print("titles after filters:", len(news), "| tickers:", news["ticker"].nunique())

    # ---- trading calendar = days on which FINRA published a Nasdaq-TRF file ----
    tdays = np.array(sorted({int(os.path.basename(p)[4:12]) for p in glob.glob(f"{RAW}/regsho/FNSQ*.txt")}))
    print("trading days:", len(tdays), tdays[0], "->", tdays[-1])
    nd = news["news_date"].to_numpy()
    i_prev = np.searchsorted(tdays, nd, "left") - 1          # last trading day strictly before the news date
    i_next = np.searchsorted(tdays, nd, "right")             # first trading day strictly after the news date
    ok = (i_prev >= 0) & (i_next < len(tdays))
    news = news[ok].copy()
    news["d_prev"], news["d_next"] = tdays[i_prev[ok]], tdays[i_next[ok]]

    # ---- FINRA daily short-sale volume, read only for the (day, ticker) pairs we need ----
    need = pd.concat([news[["d_prev", "ticker"]].set_axis(["d", "t"], axis=1),
                      news[["d_next", "ticker"]].set_axis(["d", "t"], axis=1)]).drop_duplicates()
    need = need.groupby("d")["t"].agg(set).to_dict()
    frames = []
    for path in sorted(glob.glob(f"{RAW}/regsho/*.txt")):
        day = int(os.path.basename(path)[4:12])
        if day not in need:
            continue
        with open(path) as fh:
            first = fh.readline()
        if first.startswith("Date|"):
            df = pd.read_csv(path, sep="|", dtype={"Symbol": str}, quoting=3)
        else:   # a few FINRA files ship without a header row
            cols = ["Date", "Symbol", "ShortVolume", "TotalVolume", "Market"] if first.count("|") == 4 else \
                   ["Date", "Symbol", "ShortVolume", "ShortExemptVolume", "TotalVolume", "Market"]
            df = pd.read_csv(path, sep="|", dtype={"Symbol": str}, header=None, names=cols, quoting=3)
        df = df[["Date", "Symbol", "ShortVolume", "TotalVolume"]].dropna(subset=["Symbol"])   # drops the record-count trailer
        frames.append(df[df["Symbol"].isin(need[day])])
    sv = pd.concat(frames).groupby(["Date", "Symbol"], as_index=False)[["ShortVolume", "TotalVolume"]].sum()
    sv["Date"] = sv["Date"].astype(int)
    key = sv.set_index(["Date", "Symbol"])
    a = key.reindex(pd.MultiIndex.from_arrays([news["d_prev"], news["ticker"]]))
    b = key.reindex(pd.MultiIndex.from_arrays([news["d_next"], news["ticker"]]))
    news["sv_prev"], news["sv_next"] = a["ShortVolume"].to_numpy(), b["ShortVolume"].to_numpy()
    diff = news["sv_next"] - news["sv_prev"]
    news["label_regsho_volume"] = np.where(diff > 0, "increase", np.where(diff < 0, "decrease", None))
    news.loc[diff.isna(), "label_regsho_volume"] = None
    zero_share = float((diff == 0).sum() / diff.notna().sum())
    print(f"regsho: S(t+1)-S(t-1)==0 for {zero_share:.2%} of matched titles (paper: <0.5%)")

    # ---- FINRA consolidated short interest ----
    si_frames = []
    for path in sorted(glob.glob(f"{RAW}/si/shrt*.csv")):
        df = pd.read_csv(path, sep="|", dtype={"symbolCode": str}, quoting=3,
                         usecols=["symbolCode", "issueName", "marketClassCode", "currentShortPositionQuantity", "settlementDate"])
        si_frames.append(df[df["marketClassCode"] != "OTC"])
    si = pd.concat(si_frames)
    si["sd"] = si["settlementDate"].str.replace("-", "").astype(int)
    names = si.sort_values("sd").groupby("symbolCode")["issueName"].last()
    sdates = np.array(sorted(si["sd"].unique()))
    si_key = si.drop_duplicates(["sd", "symbolCode"]).set_index(["sd", "symbolCode"])["currentShortPositionQuantity"]
    j = np.searchsorted(tdays, news["news_date"].to_numpy(), "left")        # first trading day >= news date
    settle = np.where(j + 2 < len(tdays), tdays[np.minimum(j + 2, len(tdays) - 1)], 0)
    k = np.searchsorted(sdates, settle, "left")
    ok = (settle > 0) & (k >= 1) & (k < len(sdates)) & (news["news_date"].to_numpy() >= sdates[0])
    s_after = np.where(ok, sdates[np.minimum(k, len(sdates) - 1)], 0)
    s_before = np.where(ok, sdates[np.maximum(k - 1, 0)], 0)
    news["si_before_date"], news["si_after_date"] = s_before, s_after
    news["si_before"] = si_key.reindex(pd.MultiIndex.from_arrays([s_before, news["ticker"]])).to_numpy()
    news["si_after"] = si_key.reindex(pd.MultiIndex.from_arrays([s_after, news["ticker"]])).to_numpy()
    news.loc[~ok, ["si_before", "si_after"]] = np.nan
    d2 = news["si_after"] - news["si_before"]
    news["label_short_interest"] = np.where(d2 > 0, "increase", np.where(d2 < 0, "decrease", None))
    news.loc[d2.isna(), "label_short_interest"] = None

    # company name for the prompt: FINRA issue name where available, else the ticker itself
    news["company"] = news["ticker"].map(names).fillna(news["ticker"])
    keep = ["title", "ticker", "company", "date", "news_date", "n_tokens", "d_prev", "d_next", "sv_prev", "sv_next",
            "label_regsho_volume", "si_before_date", "si_after_date", "si_before", "si_after", "label_short_interest"]
    out = news[keep].reset_index(drop=True)
    out.to_parquet(PREBUILT_PATH, index=False)
    print("saved", PREBUILT_PATH)
    return out

def find_prebuilt():
    for p in [PREBUILT_PATH, "/content/sitp_dataset.parquet", "/content/drive/MyDrive/sitp_dataset.parquet"]:
        if os.path.exists(p):
            return p
    try:   # Colab: ask for the file if it has not been uploaded yet
        from google.colab import files  # type: ignore
        print("Upload sitp_dataset.parquet (from the Stage 1 deliverables):")
        up = files.upload()
        name = next(iter(up))
        os.replace(name, PREBUILT_PATH)
        return PREBUILT_PATH
    except ImportError:
        return None

if DATA_MODE == "rebuild" and not os.path.exists(PREBUILT_PATH):
    full = build_dataset()
else:
    path = find_prebuilt()
    if path is None:
        raise FileNotFoundError("sitp_dataset.parquet not found; upload it or set DATA_MODE='rebuild'")
    full = pd.read_parquet(path)
label_col = f"label_{LABEL_SET}"
data = full[full[label_col].notna()].rename(columns={label_col: "label"}).reset_index(drop=True)
print(f"{LABEL_SET}: {len(data):,} labeled titles, {data['news_date'].min()} → {data['news_date'].max()}")

## 4. Stratified 80/20 split and dataset statistics (paper Section 4)

In [ ]:
train_df, test_df = train_test_split(data, test_size=0.2, stratify=data["label"], random_state=SEED)
train_df, test_df = train_df.reset_index(drop=True), test_df.reset_index(drop=True)

def vocab(s):
    return len(set(w for t in s for w in t.split()))

stats = pd.DataFrame({
    "paper": ["136,192", "54.7%", "45.3%", "16.53", "16.48", "66,624", "30,787"],
    "ours": [f"{len(data):,}", f"{(data.label == 'increase').mean():.1%}", f"{(data.label == 'decrease').mean():.1%}",
             f"{train_df.n_tokens.mean():.2f}", f"{test_df.n_tokens.mean():.2f}",
             f"{vocab(train_df.title):,}", f"{vocab(test_df.title):,}"]},
    index=["# titles", "% increase", "% decrease", "avg tokens (train)", "avg tokens (test)",
           "unique tokens (train, whitespace)", "unique tokens (test, whitespace)"])
print(stats.to_string())
stats.to_csv(f"{OUT_DIR}/dataset_stats.csv")

# Evaluation sample: stratified draw of N_TEST titles from the test split (same class ratio as the full test set)
if N_TEST and N_TEST < len(test_df):
    eval_df, _ = train_test_split(test_df, train_size=N_TEST, stratify=test_df["label"], random_state=SEED)
    eval_df = eval_df.reset_index(drop=True)
else:
    eval_df = test_df
print("evaluation sample:", len(eval_df), eval_df.label.value_counts().to_dict())

## 5. Metrics and baselines
Table 1 reports per-class F1 for `increase` and `decrease` plus an "Average" column described as the weighted average F1.
We report per-class F1, support-weighted F1 and macro F1. Unparseable LLM outputs count as wrong.
The paper only reports a majority baseline; we add a random baseline (predict each class with its training prior),
because with these class shares a random guesser already reaches weighted F1 ≈ 0.50.

In [ ]:
LABELS = ["increase", "decrease"]

def score(y_true, y_pred):
    per = f1_score(y_true, y_pred, labels=LABELS, average=None, zero_division=0)
    return {"f1_increase": round(float(per[0]), 4), "f1_decrease": round(float(per[1]), 4),
            "f1_weighted": round(float(f1_score(y_true, y_pred, labels=LABELS, average="weighted", zero_division=0)), 4),
            "f1_macro": round(float(f1_score(y_true, y_pred, labels=LABELS, average="macro", zero_division=0)), 4),
            "accuracy": round(float(np.mean(np.array(y_true) == np.array(y_pred))), 4),
            "invalid_rate": round(float(np.mean(~pd.Series(y_pred).isin(LABELS))), 4),
            "n": int(len(y_true))}

rng = np.random.default_rng(SEED)
majority = train_df.label.value_counts().idxmax()
p_inc = (train_df.label == "increase").mean()
baselines = {
    "majority (full test)": score(test_df.label, [majority] * len(test_df)),
    "majority (eval sample)": score(eval_df.label, [majority] * len(eval_df)),
    "random prior (full test)": score(test_df.label, np.where(rng.random(len(test_df)) < p_inc, "increase", "decrease")),
    "random prior (eval sample)": score(eval_df.label, np.where(rng.random(len(eval_df)) < p_inc, "increase", "decrease")),
}
print(pd.DataFrame(baselines).T.to_string())
json.dump(baselines, open(f"{OUT_DIR}/baselines.json", "w"), indent=2)

## 6. Prompt (paper Figure 4) and in-context examples
In-context examples are drawn at random from the **training** split (fixed seed per test title, so every model sees the
same demonstrations). Each demonstration is the full Figure 4 prompt followed by its gold answer.

In [ ]:
TEMPLATE = ("Read the news title below and determine how the short interest of {company} changes.\n\n"
            "{title}\n\n"
            "OPTIONS:\n1. increase\n2. decrease\n\n"
            "ANSWER:")

def render(row, answer=None):
    p = TEMPLATE.format(company=row["company"], title=row["title"])
    return p + (" " + answer if answer else "")

def build_prompt(row, k, idx):
    if k == 0:
        return render(row)
    r = np.random.default_rng(SEED * 1_000_003 + idx)
    demos = train_df.iloc[r.choice(len(train_df), size=k, replace=False)]
    return "\n\n".join([render(d, d["label"]) for _, d in demos.iterrows()] + [render(row)])

print(build_prompt(eval_df.iloc[0], 1, 0))

# Save the full experiment configuration next to the results
config = {
    "paper": "Xiao et al., Short Interest Trend Prediction with Large Language Models, ICIAI 2024, doi:10.1145/3655497.3655500",
    "label_set": LABEL_SET,
    "label_definition": {
        "regsho_volume": "sign(S[t+1]-S[t-1]); S = FINRA Reg SHO daily short-sale volume (TRFs + ADF); paper-faithful, NOT short interest",
        "short_interest": "sign(SI[first settlement >= T+2 of news day] - SI[previous settlement]); FINRA consolidated short interest",
    }[LABEL_SET],
    "dataset_sources": {
        "news_titles": "Kaggle miguelaenlle/massive-stock-news-analysis-db-for-nlpbacktests, analyst_ratings_processed.csv (Benzinga)",
        "regsho_volume": "https://cdn.finra.org/equity/regsho/daily/ (FNSQ, FNYX, FNQC, FNRA files)",
        "short_interest": "https://cdn.finra.org/equity/otcmarket/biweekly/ (consolidated short interest, from 2017-12-29)",
    },
    "filters": "ASCII-only titles, 1-50 FLAN-T5 tokens, exact (title, ticker, date) duplicates removed",
    "n_labeled": int(len(data)), "n_train": int(len(train_df)), "n_test_full": int(len(test_df)),
    "n_test_evaluated": int(len(eval_df)), "split": "stratified 80/20, random_state=SEED",
    "seed": SEED, "model_name": MODEL_NAME, "load_8bit": LOAD_8BIT, "scoring": SCORING,
    "shots": SHOTS, "demo_selection": "random from training split, fixed per test title (seed = SEED*1000003 + index)",
    "prompt_template": TEMPLATE, "demo_format": "full template + ' <gold label>', demos joined by blank lines",
    "decoding": "greedy, max_new_tokens=5, first of increase/decrease/1./2. in output; otherwise invalid (counted wrong)",
    "max_input_tokens": MAX_INPUT_LEN, "truncation_side": "left", "batch_size": BATCH_SIZE,
    "metrics": "per-class F1, support-weighted F1, macro F1 (sklearn, labels=[increase, decrease])",
    "transformers_version": transformers.__version__, "torch_version": torch.__version__,
}
json.dump(config, open(f"{OUT_DIR}/experiment_config.json", "w"), indent=2)
print("saved", f"{OUT_DIR}/experiment_config.json")

## 7. Load the model

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoModelForCausalLM, AutoConfig

device = "cuda" if torch.cuda.is_available() else "cpu"
cfg = AutoConfig.from_pretrained(MODEL_NAME, token=HF_TOKEN)
IS_SEQ2SEQ = bool(getattr(cfg, "is_encoder_decoder", False))
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
if not IS_SEQ2SEQ:
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
tokenizer.truncation_side = "left"   # never cut off the question / "ANSWER:" at the end of a long few-shot prompt

kwargs = {"token": HF_TOKEN, "low_cpu_mem_usage": True}
if LOAD_8BIT:
    from transformers import BitsAndBytesConfig
    kwargs.update(quantization_config=BitsAndBytesConfig(load_in_8bit=True), device_map="auto")
elif device == "cuda":
    kwargs.update(device_map="auto")
if device == "cuda":   # bf16 on A100/L4 (best for T5-family checkpoints); fp16 on T4, which has no bf16 support
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
else:
    dtype = torch.float32
Auto = AutoModelForSeq2SeqLM if IS_SEQ2SEQ else AutoModelForCausalLM
try:
    model = Auto.from_pretrained(MODEL_NAME, dtype=dtype, **kwargs)
except TypeError:                                   # older transformers
    model = Auto.from_pretrained(MODEL_NAME, torch_dtype=dtype, **kwargs)
if device == "cpu" and not LOAD_8BIT:
    torch.set_num_threads(os.cpu_count())
model.eval()
print(MODEL_NAME, "| seq2seq:", IS_SEQ2SEQ, "| device:", model.device)

## 8. Inference
`generate` follows the paper (greedy completion after "ANSWER:", then parse the first option word).
`rank` scores both options and picks the more likely one (never invalid; useful as a diagnostic).
Predictions are checkpointed to CSV after every batch, so a disconnected Colab session resumes where it stopped.

In [ ]:
def parse(text):
    t = text.strip().lower()
    hits = [(t.find(w), lab) for w, lab in [("increase", "increase"), ("decrease", "decrease"),
                                             ("1.", "increase"), ("2.", "decrease")] if t.find(w) >= 0]
    if not hits and t[:1] in "12":
        return "increase" if t[0] == "1" else "decrease"
    return min(hits)[1] if hits else "invalid"

@torch.no_grad()
def predict_generate(prompts):
    enc = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=MAX_INPUT_LEN).to(model.device)
    out = model.generate(**enc, max_new_tokens=5, do_sample=False, pad_token_id=tokenizer.pad_token_id)
    if not IS_SEQ2SEQ:
        out = out[:, enc["input_ids"].shape[1]:]
    texts = tokenizer.batch_decode(out, skip_special_tokens=True)
    return [parse(t) for t in texts], texts

@torch.no_grad()
def predict_rank(prompts):
    preds, texts = [], []
    enc = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=MAX_INPUT_LEN).to(model.device)
    scores = []
    for opt in LABELS:
        if IS_SEQ2SEQ:
            lab = tokenizer([opt] * len(prompts), return_tensors="pt", padding=True).input_ids.to(model.device)
            logits = model(**enc, labels=lab).logits
            lp = torch.log_softmax(logits.float(), -1).gather(-1, lab.unsqueeze(-1)).squeeze(-1)
            lp = (lp * (lab != tokenizer.pad_token_id)).sum(-1)
        else:
            opt_ids = tokenizer(" " + opt, add_special_tokens=False).input_ids
            ids = torch.cat([enc.input_ids, torch.tensor([opt_ids] * len(prompts), device=model.device)], 1)
            att = torch.cat([enc.attention_mask, torch.ones(len(prompts), len(opt_ids), device=model.device, dtype=enc.attention_mask.dtype)], 1)
            logits = model(input_ids=ids, attention_mask=att).logits[:, -len(opt_ids) - 1:-1]
            lp = torch.log_softmax(logits.float(), -1).gather(-1, torch.tensor([opt_ids] * len(prompts), device=model.device).unsqueeze(-1)).squeeze(-1).sum(-1)
        scores.append(lp)
    s = torch.stack(scores, 1)
    for row in s.tolist():
        preds.append(LABELS[int(np.argmax(row))]); texts.append(f"lp_inc={row[0]:.3f} lp_dec={row[1]:.3f}")
    return preds, texts

predict = predict_generate if SCORING == "generate" else predict_rank

def run_setting(k):
    path = f"{OUT_DIR}/predictions_{k}shot.csv"
    done = pd.read_csv(path) if os.path.exists(path) else pd.DataFrame()
    start = len(done)
    rows = done.to_dict("records")
    t0 = time.time()
    for b in range(start, len(eval_df), BATCH_SIZE):
        batch = eval_df.iloc[b:b + BATCH_SIZE]
        prompts = [build_prompt(r, k, b + i) for i, (_, r) in enumerate(batch.iterrows())]
        preds, texts = predict(prompts)
        for (_, r), p, t, pr in zip(batch.iterrows(), preds, texts, prompts):
            rows.append({"idx": len(rows), "ticker": r["ticker"], "news_date": r["news_date"], "title": r["title"],
                         "gold": r["label"], "pred": p, "raw_output": t,
                         "prompt_tokens": len(tokenizer(pr).input_ids)})
        pd.DataFrame(rows).to_csv(path, index=False)
        if (b // BATCH_SIZE) % 10 == 0:
            el = time.time() - t0
            print(f"  {k}-shot: {len(rows)}/{len(eval_df)}  ({el / max(1, len(rows) - start):.2f}s/ex)", flush=True)
    return pd.DataFrame(rows)

results = {}
for k in SHOTS:
    print(f"== {k}-shot ==")
    preds = run_setting(k)
    results[f"{k}-shot"] = score(preds["gold"], preds["pred"])
    results[f"{k}-shot"]["max_prompt_tokens"] = int(preds["prompt_tokens"].max())
    results[f"{k}-shot"]["truncated"] = int((preds["prompt_tokens"] > MAX_INPUT_LEN).sum())
    print(results[f"{k}-shot"])
    json.dump(results, open(f"{OUT_DIR}/llm_results.json", "w"), indent=2)

## 9. Results in Table 1 format, next to the paper's numbers
**Paper numbers are copied from Table 1 of Xiao et al. (2024); they were not produced by this notebook.**

In [ ]:
PAPER = {  # Table 1 of the paper: (increase F1, decrease F1, "Average") for 0/1/5/10-shot
    "Majority": {"all": (0.71, 0.00, 0.35)},
    "FLAN-T5 (11B)": {0: (0.03, 0.62, 0.30), 1: (0.23, 0.59, 0.40), 5: (0.30, 0.60, 0.38), 10: (0.31, 0.59, 0.44)},
    "LLaMA 2 (7B chat)": {0: (0.17, 0.40, 0.28), 1: (0.06, 0.17, 0.08), 5: (0.00, 0.14, 0.10), 10: (0.00, 0.31, 0.39)},
    "FLAN-UL2 (20B)": {0: (0.18, 0.61, 0.37), 1: (0.16, 0.61, 0.36), 5: (0.34, 0.58, 0.45), 10: (0.39, 0.56, 0.47)},
}
rows = []
for m, d in PAPER.items():
    for k, (fi, fd, avg) in d.items():
        # support-weighted F1 recomputed from the paper's own per-class F1 and class shares (54.7% / 45.3%)
        rows.append({"source": "paper (reported)", "model": m, "shots": k, "f1_increase": fi, "f1_decrease": fd,
                     "f1_avg_reported": avg, "f1_weighted_recomputed": round(0.547 * fi + 0.453 * fd, 3)})
b = baselines["majority (eval sample)"]
rows.append({"source": "ours (measured)", "model": "Majority", "shots": "all", **{x: b[x] for x in ["f1_increase", "f1_decrease", "f1_weighted", "f1_macro"]}})
b = baselines["random prior (eval sample)"]
rows.append({"source": "ours (measured)", "model": "Random (class prior)", "shots": "all", **{x: b[x] for x in ["f1_increase", "f1_decrease", "f1_weighted", "f1_macro"]}})
for k, r in results.items():
    rows.append({"source": "ours (measured)", "model": MODEL_NAME, "shots": int(k.split("-")[0]),
                 **{x: r[x] for x in ["f1_increase", "f1_decrease", "f1_weighted", "f1_macro", "invalid_rate"]}})
table = pd.DataFrame(rows)
print(f"Label set: {LABEL_SET} | eval n = {len(eval_df)} | scoring = {SCORING}")
print(table.to_string(index=False))
table.to_csv(f"{OUT_DIR}/comparison_table.csv", index=False)

## 10. Error analysis: confusion matrices and prediction distribution

In [ ]:
for k in SHOTS:
    p = pd.read_csv(f"{OUT_DIR}/predictions_{k}shot.csv")
    cm = confusion_matrix(p["gold"], p["pred"], labels=LABELS + ["invalid"])[:2]
    print(f"\n{k}-shot  rows=gold, cols=pred [increase, decrease, invalid]\n{cm}")
    print("  predicted share:", p["pred"].value_counts(normalize=True).round(3).to_dict())

## 11. Save / download
All outputs are in `OUT_DIR`: `predictions_{k}shot.csv`, `llm_results.json`, `baselines.json`, `dataset_stats.csv`,
`comparison_table.csv`. In Colab, zip and download them:

In [ ]:
import shutil
archive = shutil.make_archive(OUT_DIR, "zip", OUT_DIR)
print("Saved:", archive)
try:
    from google.colab import files  # type: ignore
    files.download(archive)
except Exception:
    pass